# Bài học 12 - Giảm lịch sử trò chuyện với Vở ghi chú của tác nhân

Sổ ghi chép này trình bày cách quản lý ngữ cảnh trong các cuộc trò chuyện dài sử dụng Microsoft Agent Framework. Khi các cuộc trò chuyện tăng lên, số lượng token cũng tăng — cuối cùng vượt quá cửa sổ ngữ cảnh của mô hình. Chúng ta giải quyết điều này bằng **mẫu tóm tắt ngữ cảnh** và **vở ghi chú của tác nhân** để lưu giữ bộ nhớ lâu dài.

## Những gì bạn sẽ học:
1. **Tại sao quản lý ngữ cảnh quan trọng**: Hiểu về giới hạn token và cửa sổ ngữ cảnh
2. **Tác nhân nhận biết ngữ cảnh**: Xây dựng các tác nhân quản lý ngữ cảnh cuộc trò chuyện riêng của chúng
3. **Mẫu tóm tắt ngữ cảnh**: Sử dụng công cụ để cô đọng lịch sử cuộc trò chuyện
4. **Vở ghi chú của tác nhân**: Bộ nhớ bền vững tồn tại qua quá trình giảm ngữ cảnh

## Yêu cầu trước:
- Cài đặt Azure OpenAI với các biến môi trường đã được cấu hình
- Hiểu biết cơ bản về các khái niệm tác nhân từ các bài học trước đó


## Cài đặt


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Tại sao Quản lý Ngữ cảnh lại Quan trọng

Mỗi LLM có một **cửa sổ ngữ cảnh** hữu hạn — số lượng token tối đa mà nó có thể xử lý trong một yêu cầu đơn. Khi một cuộc hội thoại nhiều lượt tiến triển:

- **Số lượng token tăng tuyến tính** với mỗi tin nhắn người dùng và phản hồi của trợ lý.
- **Token trong prompt chiếm phần lớn chi phí** vì toàn bộ lịch sử được gửi lại mỗi lượt.
- Cuối cùng cuộc hội thoại **vượt quá cửa sổ ngữ cảnh** và mô hình sẽ cắt ngắn hoặc báo lỗi.

### Chiến lược Quản lý Ngữ cảnh

| Chiến lược | Cách Hoạt Động | Đánh đổi |
|---|---|---|
| **Cắt ngắn** | Bỏ các tin nhắn cũ nhất | Mất ngữ cảnh ban đầu |
| **Tóm tắt** | Cô đọng các tin nhắn cũ thành một bản tóm tắt | Mất một số chi tiết, nhưng giữ lại các điểm chính |
| **Scratchpad / Bộ nhớ Ngoài** | Lưu giữ các thông tin quan trọng bên ngoài cuộc hội thoại | Cần gọi công cụ, nhưng tồn tại khi có bất kỳ giảm bớt nào |

Trong sổ tay này chúng ta kết hợp **tóm tắt** với một **công cụ scratchpad** để đại lý có thể duy trì tính liên tục ngay cả khi lịch sử cuộc hội thoại bị cô đọng.


## Tạo một Tác nhân Nhận thức ngữ cảnh


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Mô phỏng một Cuộc hội thoại Dài

Hãy cùng trải qua một cuộc hội thoại đa lượt để xem cách ngữ cảnh tích lũy. Đại lý nên giữ lại các chi tiết chính (sở thích, ngân sách, ngày đi du lịch) qua các lượt và thể hiện sự liền mạch.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Lưu ý cách mà tác nhân giữ ngữ cảnh từ các lượt trước — nó biết về Nhật Bản, sushi, đền chùa, nhiếp ảnh, ngân sách 3000 đô la, du lịch một mình, và khoảng thời gian tháng Tư. Trong một cuộc trò chuyện ngắn, điều này hoạt động tốt, nhưng khi cuộc trò chuyện kéo dài, việc gửi lại toàn bộ lịch sử trở nên tốn kém.

Hãy tiếp tục cuộc trò chuyện với nhiều lượt hơn để xem sự tích lũy ngữ cảnh:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Mẫu Tóm Tắt Bối Cảnh

Khi cuộc trò chuyện phát triển, chúng ta có thể sử dụng một **công cụ tóm tắt** để cô đọng bối cảnh đã tích lũy thành một định dạng ngắn gọn. Đại lý gọi công cụ này để ghi lại các sở thích chính để ngay cả khi các tin nhắn cũ hơn bị bỏ qua, thông tin quan trọng vẫn được bảo tồn.

Mẫu này là nền tảng cho việc giảm lịch sử tinh vi hơn:
1. Đại lý xác định các sự kiện chính từ cuộc trò chuyện
2. Nó gọi công cụ tóm tắt để ghi lại chúng
3. Các tin nhắn cũ hơn có thể được loại bỏ an toàn vì bản tóm tắt đã ghi lại những điều quan trọng

Dưới đây chúng tôi định nghĩa một công cụ `summarize_preferences` mà đại lý có thể gọi để ghi lại bản tóm tắt ngắn gọn về những gì nó đã học được.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Tóm tắt

Trong bài học này, bạn đã học cách quản lý ngữ cảnh trong các cuộc trò chuyện tác nhân kéo dài bằng cách sử dụng Microsoft Agent Framework:

### Khái niệm chính
- **Cửa sổ ngữ cảnh là hữu hạn** — mỗi token trong lịch sử cuộc trò chuyện đều tốn chi phí và được tính vào giới hạn.
- **Công cụ tóm tắt** cho phép tác nhân cô đọng ngữ cảnh tích lũy thành các bản tóm tắt ngắn gọn, giảm sử dụng token trong khi vẫn giữ thông tin trọng yếu.
- **Bảng ghi chú tác nhân** cung cấp bộ nhớ bên ngoài bền bỉ, tồn tại qua mọi việc giảm bớt cuộc trò chuyện.

### Những gì bạn đã xây dựng
- Một **tác nhân nhận biết ngữ cảnh** duy trì sự liên tục xuyên suốt cuộc trò chuyện nhiều lượt
- Một **công cụ tóm tắt** (`summarize_preferences`) ghi lại các chi tiết chính của người dùng dưới định dạng gọn nhẹ
- Một **cuộc trò chuyện nhiều lượt** minh họa cho việc giữ ngữ cảnh và xử lý thay đổi

### Ứng dụng trong thế giới thực
- **Bot Dịch vụ Khách hàng**: Ghi nhớ sở thích qua nhiều phiên hỗ trợ dài
- **Trợ lý Cá nhân**: Theo dõi dự án đang diễn ra mà không cần giải thích lại ngữ cảnh
- **Gia sư Giáo dục**: Duy trì tiến trình học tập của học sinh qua nhiều lần tương tác

### Các bước tiếp theo
- Triển khai công cụ bảng ghi chú đầy đủ với khả năng lưu trữ dựa trên file
- Thêm việc tự động cắt ngắn lịch sử sau khi tóm tắt
- Kết hợp với cơ sở dữ liệu vector để tìm kiếm bộ nhớ ngữ nghĩa
- Xây dựng các tác nhân có thể tiếp tục cuộc trò chuyện sau nhiều ngày với đầy đủ ngữ cảnh


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc không chính xác. Văn bản gốc bằng ngôn ngữ gốc nên được coi là nguồn tham khảo chính thức. Đối với những thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
